In [ ]:
import os
import pandas as pd
import numpy as np

from src.synthetic_control_batch import benchmark_sweep

# Load data

In [ ]:
data_dir = os.path.join('..', '..', 'data')
syn_control_dir = os.path.join(data_dir, 'MovieLens-20M', 'synthetic_control')

real_df = pd.read_csv(os.path.join(syn_control_dir, 'real.csv'), index_col=0)
llm_df = pd.read_csv(os.path.join(syn_control_dir, 'LLM.csv'), index_col=0)

specs = [{
    'name': 'MovieLens-20M',
    'real': real_df.to_numpy(),
    'synthetic': llm_df.to_numpy(),
}]

print(f'Loaded {len(specs)} specification(s)')
for s in specs:
    r = s['real']
    pct_missing = np.isnan(r).sum() / r.size * 100
    print(f"  {s['name']}: {r.shape}  ({pct_missing:.1f}% missing)")

output_dir = os.path.join('..', '..', 'outputs', 'synthetic_control', 'MovieLens_batch')
os.makedirs(output_dir, exist_ok=True)

# Define methods

In [ ]:
# Best column-wise parameters
methods_column = [
    {'label': 'Ridge',                'method': 'ridge',                  'regularization_multiplier': 1000},
    {'label': 'Lasso',                'method': 'lasso',                  'regularization_multiplier': 0.01},
    {'label': 'Elastic Net',          'method': 'elastic_net',            'regularization_multiplier': 0.1, 'en_l1_ratio': 0.1},
    {'label': 'Synthetic Control',    'method': 'synthetic_control',      'regularization_multiplier': 1e-8},
    {'label': 'Neural Net',           'method': 'neural_net',             'nn_hidden_dims': [16], 'nn_epochs': 200, 'nn_lr': 1e-3, 'nn_weight_decay': 0.1, 'nn_batch_size': 128, 'nn_patience': 20, 'nn_seed': 42},
    {'label': 'MC Hard SVD',          'method': 'mc_hard_svd',            'mc_rank': 5},
    {'label': 'MC Soft SVD',          'method': 'mc_soft_svd',            'mc_rank': 15, 'mc_lambda': 5},
    {'label': 'MC ALS',               'method': 'mc_als',                 'mc_rank': 15, 'mc_lambda': 5},
    {'label': 'MC Synthetic Prior',   'method': 'mc_synthetic_prior',     'mc_rank': 8},
    {'label': 'Synthetic Intervention',  'method': 'synthetic_intervention', 'si_rank': 50, 'regularization_multiplier': 100},
]

# Best row-wise parameters
methods_row = [
    {'label': 'Ridge',                'method': 'ridge',                  'regularization_multiplier': 1000},
    {'label': 'Lasso',                'method': 'lasso',                  'regularization_multiplier': 0.1},
    {'label': 'Elastic Net',          'method': 'elastic_net',            'regularization_multiplier': 1, 'en_l1_ratio': 0.1},
    {'label': 'Synthetic Control',    'method': 'synthetic_control',      'regularization_multiplier': 1},
    {'label': 'Neural Net',           'method': 'neural_net',             'nn_hidden_dims': [8], 'nn_epochs': 200, 'nn_lr': 1e-3, 'nn_weight_decay': 1, 'nn_batch_size': 128, 'nn_patience': 20, 'nn_seed': 42},
    {'label': 'MC Hard SVD',          'method': 'mc_hard_svd',            'mc_rank': 2},
    {'label': 'MC Soft SVD',          'method': 'mc_soft_svd',            'mc_rank': 5, 'mc_lambda': 20},
    {'label': 'MC ALS',               'method': 'mc_als',                 'mc_rank': 3, 'mc_lambda': 5},
    {'label': 'MC Synthetic Prior',   'method': 'mc_synthetic_prior',     'mc_rank': 2},
    {'label': 'Synthetic Intervention',  'method': 'synthetic_intervention', 'si_rank': 10, 'regularization_multiplier': 100},
]

# Column-wise benchmark

In [ ]:
results_col = benchmark_sweep(
    specs=specs,
    methods=methods_column,
    direction='column',
    sc_kwargs={'imputation_rank': 5, 'min_col_std': 1},
    n_jobs=-2,
    verbose=False,
)

# Column-wise results

In [ ]:
results_col['formatted']

In [ ]:
results_col['correlation_mean']

# Save results

In [ ]:
results_col['correlation_mean'].to_csv(os.path.join(output_dir, 'column_corr_mean.csv'))
results_col['correlation_se'].to_csv(os.path.join(output_dir, 'column_corr_se.csv'))
results_col['formatted'].to_csv(os.path.join(output_dir, 'column_formatted.csv'))

# Row-wise benchmark

In [ ]:
results_row = benchmark_sweep(
    specs=specs,
    methods=methods_row,
    direction='row',
    sc_kwargs={'imputation_rank': 5, 'min_col_std': 1},
    n_jobs=-2,
    verbose=False,
)

# Row-wise results

In [ ]:
results_row['formatted']

In [ ]:
results_row['correlation_mean']

# Save results

In [ ]:
results_row['correlation_mean'].to_csv(os.path.join(output_dir, 'row_corr_mean.csv'))
results_row['correlation_se'].to_csv(os.path.join(output_dir, 'row_corr_se.csv'))
results_row['formatted'].to_csv(os.path.join(output_dir, 'row_formatted.csv'))